# FujiCV — CIFAR-10 Training on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dsabarinathan/fujicv/blob/main/examples/colab_cifar10.ipynb)

Trains **ResNet-18** on **CIFAR-10** using the `fujicv` package.  
Uses **torchvision on-the-fly augmentation** — no slow external downloads.  
All results saved to `/content/results/`.

> **Tip:** Set runtime to GPU → Runtime → Change runtime type → **T4 GPU**

## 1. Install FujiCV

In [ ]:
!pip install fujicv timm -q
print('Done.')

## 2. Imports & config

In [ ]:
import logging
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import fujicv
from fujicv.engine.trainer import Trainer
from fujicv.eval.curves import plot_pr_curve, plot_roc_curve
from fujicv.eval.plots import plot_loss_curves, plot_metric_curves
from fujicv.eval.report import classification_report
from fujicv.losses.classification import CrossEntropyLoss
from fujicv.metrics.classification import Accuracy, F1
from fujicv.models.builder import ModelBuilder
from fujicv.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

SEED       = 42
IMAGE_SIZE = 32
BATCH_SIZE = 256
EPOCHS     = 10
LR         = 1e-3
OUTPUT_DIR = '/content/results'

os.makedirs(OUTPUT_DIR, exist_ok=True)
set_seed(SEED)
device = fujicv.get_device()
print(f'Device: {device}')

## 3. Load CIFAR-10 with on-the-fly torchvision augmentation
Uses torchvision's built-in CIFAR-10 loader — fast CDN download (~170 MB, ~1 min on Colab).  
Augmentation applied on-the-fly every epoch via `transforms.Compose`.

In [ ]:
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

MEAN = (0.4914, 0.4822, 0.4465)   # CIFAR-10 statistics
STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.CIFAR10(root='/content/data', train=True,  download=True, transform=train_transform)
val_ds   = datasets.CIFAR10(root='/content/data', train=False, download=True, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Classes: {CLASS_NAMES}')

## 4. Preview sample batch

In [ ]:
imgs, labels = next(iter(val_loader))
mean_t = torch.tensor(MEAN).view(3,1,1)
std_t  = torch.tensor(STD).view(3,1,1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img = imgs[i] * std_t + mean_t
    ax.imshow(img.permute(1,2,0).clamp(0,1).numpy())
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('CIFAR-10 — Sample Batch', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_batch.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Build model

In [ ]:
model = ModelBuilder(
    backbone_name='resnet18',
    backbone_source='timm',
    pretrained=True,
    task='classification',
    num_outputs=10,
    image_size=IMAGE_SIZE,
).build()

total = sum(p.numel() for p in model.parameters()) / 1e6
print(f'ResNet-18 | {total:.1f}M parameters')

## 6. Train

In [ ]:
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=CrossEntropyLoss(),
    metrics={'accuracy': Accuracy(), 'f1': F1()},
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    task='classification',
    output_dir=OUTPUT_DIR,
    class_to_idx=class_to_idx,
    monitor_metric='val_accuracy',
    mixed_precision=True,
    early_stopping_patience=5,
)

history = trainer.train()

## 7. Training curves

In [ ]:
for metric, fname in [('loss','loss_curves'), ('accuracy','accuracy_curves'), ('f1','f1_curves')]:
    fig = plot_loss_curves(history) if metric == 'loss' else plot_metric_curves(history, metric)
    fig.savefig(f'{OUTPUT_DIR}/{fname}.png', dpi=120, bbox_inches='tight')
    plt.show()
print('Saved loss, accuracy, f1 curves.')

## 8. Evaluate on validation set

In [ ]:
model.eval()
all_preds, all_probs, all_targets = [], [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        logits = model(imgs.to(device))
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_targets.append(labels.numpy())

all_probs   = np.concatenate(all_probs)
all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

acc = (all_preds == all_targets).mean()
print(f'Val Accuracy: {acc*100:.2f}%')

## 9. Classification report & confusion matrix

In [ ]:
report_text, fig_cm = classification_report(all_targets, all_preds, CLASS_NAMES)
print(report_text)
with open(f'{OUTPUT_DIR}/classification_report.txt', 'w') as f:
    f.write(report_text)
fig_cm.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. ROC & PR curves

In [ ]:
plot_roc_curve(all_targets, all_probs, CLASS_NAMES, multi_class='ovr').savefig(
    f'{OUTPUT_DIR}/roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

plot_pr_curve(all_targets, all_probs, CLASS_NAMES).savefig(
    f'{OUTPUT_DIR}/pr_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. t-SNE visualization

In [ ]:
from fujicv.eval.tsne import extract_embeddings, plot_tsne

subset = torch.utils.data.Subset(val_ds, range(2000))
sub_loader = DataLoader(subset, batch_size=256, num_workers=2)

embeddings, embed_labels = extract_embeddings(model, sub_loader, device)
plot_tsne(embeddings, embed_labels, CLASS_NAMES).savefig(
    f'{OUTPUT_DIR}/tsne.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Summary & download results

In [ ]:
best_acc = max(history.metrics.get('val_accuracy', [0]))
best_f1  = max(history.metrics.get('val_f1', [0]))

print(f"""
========================================
 FujiCV CIFAR-10 Results
========================================
 Model      : ResNet-18 (pretrained)
 Epochs     : {EPOCHS}  |  Device: {device}
 Best Val Accuracy : {best_acc*100:.2f}%
 Best Val F1       : {best_f1:.4f}
========================================
 Saved to /content/results/
   sample_batch.png   loss_curves.png
   accuracy_curves.png  f1_curves.png
   confusion_matrix.png  roc_curve.png
   pr_curve.png  tsne.png
   best.pt  history.csv
========================================
""")

# List files
for f in sorted(os.listdir(OUTPUT_DIR)):
    kb = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f'  {f:<35} {kb:6.1f} KB')

In [ ]:
# Download everything as a zip
import shutil
from google.colab import files

shutil.make_archive('/content/fujicv_cifar10_results', 'zip', OUTPUT_DIR)
files.download('/content/fujicv_cifar10_results.zip')